# Advik: TC-STAG, the method (time embeddings + time-conditioned graph mixture on DSTAGNN)
**Team project:** TC-STAG | **Owner:** Advik | **Default dataset:** PEMS04 (set `DATASET = 'PEMS08'` for the second dataset)

This is the notebook that produces **our paper's main results**. It trains our method and its ablations with exactly the same protocol as the baseline (Harshith, PEMS04 / Jinay, PEMS08).

**Variants (all are DSTAGNN + something):**
| name | time embeddings | attention-prior graph A_STRG(t) | purpose |
|---|---|---|---|
| `te` | yes | global graph (as original) | isolates the *known* effect of time features |
| `tcstag` | yes | **learned time-conditioned mixture** of global + 3 regime graphs (attention prior and Chebyshev support) | **our full method** |
| `tcstag_prior` | yes | same mixture, but only the attention prior | ablation: shows why the support must change |
| `hard` | no | regime graph picked by the clock, no learned gate | graph idea alone, no parameters |
| `soft_static` | no | learned mixture weights that do **not** depend on time | is the *time-conditioning* needed? |
| `te_hard` | yes | regime graph picked by the clock | optional |

**What the model changes (all verified on CPU against the authors' code; with every new feature off it reproduces their model's outputs to 2e-6):**
1. *Time embeddings:* time-of-day (288) and day-of-week (7) embeddings are added to the spatial-attention input and fused into the prediction head with a zero-initialised residual MLP, so training starts exactly at the baseline.
2. *Graph mixture:* the prior graph that the authors feed into the spatial attention / graph convolution (`adj_pa`) becomes `sum_c w_c(t) * A_c` with `c` in {global, weekday-peak, weekday-off-peak, weekend}; `w(t) = softmax(MLP(time embeddings))`, initialised near the global graph. The Chebyshev graph A_STAG stays global.

**IMPORTANT, what changed (v2):** (1) the graph mixture now time-conditions the **Chebyshev graph support itself** (a mix of per-regime graph filters). In the authors' code the Chebyshev matrices multiply the attention elementwise, so a time-conditioned attention *prior* alone can only reweight the ~3 edges per node that already exist (we measured: only 23% of the peak-hour prior edges fall inside that support). (2) A bug in the first version used the road-network graph instead of the learned STAG for the base model: the old `base`/`te` runs were really DSTAGNN-G (their MAE 19.41 matches the paper's DSTAGNN-G). This version fixes it (`GRAPH = 'AG'`) and `tcstag_prior` keeps the old prior-only design as an ablation.

**Before you start (Kaggle sidebar):**
1. Accelerator **GPU T4 x2**, Internet **On**.
2. Add Data: the `PEMS04` dataset (`PEMS04.npz`), **Jinay's graph dataset** (the one with `regime_graphs/`), and (optional but recommended) **Harshith's baseline output** so the notebook can compare against it automatically.
3. Use **Save & Run All (Commit)** so it survives closing the tab.

**Outputs:** `advik_tcstag_outputs.zip`, `tcstag_results_summary.csv`, `tcstag_by_regime.csv`, `gate_weights_over_day.png`.

## Cell 1: Settings (shared protocol, do not change without telling the group)

In [ ]:
import os, sys, json, time, subprocess, shutil, glob, threading, queue, zipfile
from pathlib import Path
import numpy as np, pandas as pd

DATASET     = 'PEMS04'
GRAPH_K     = ''              # '' = 1% graphs (paper's Psp). '_k5' = denser 5% graphs (try only after the main results)
MAX_EPOCHS  = 60; PATIENCE = 15          # team-wide, identical to the baseline notebooks
SESSION_HOURS_BUDGET = 10.5
RUN_SMOKE_TEST, RUN_TIMING_TEST = True, True

# WHAT THIS ACCOUNT RUNS: a list of (variant, seed). One job per GPU, so on GPU T4 x2 two jobs run in parallel (~4 h).
# Main experiment = 6 runs, split over three accounts so everything finishes in ONE round:
#     Advik   : [('tcstag', 1), ('te', 1)]        Harshith: [('tcstag', 2), ('te', 2)]        Jinay: [('tcstag', 3), ('te', 3)]
# Optional ablations later (1 seed each): 'hard', 'soft_static', 'tcstag_prior'.
JOBS = [('tcstag', 1), ('te', 1)]   # control run used JOBS = [('base', 2)]; see README
GRAPH = 'AG'                  # Chebyshev graph of the base model: 'AG' = learned STAG (the paper's DSTAGNN). Do NOT use 'G' for the main experiment.

EXPECT = {'PEMS04': (16992, 307, 3), 'PEMS08': (17856, 170, 3)}
ON_KAGGLE = os.path.exists('/kaggle')
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('./working'); WORK.mkdir(exist_ok=True, parents=True)
REPO = WORK / 'DSTAGNN'; RUNS = WORK / 'runs'; RUNS.mkdir(exist_ok=True)
SESSION_DEADLINE = time.time() + SESSION_HOURS_BUDGET * 3600
print('dataset', DATASET, '| jobs:', JOBS)

## Cell 2: Setup: authors' repo, GPUs, data, Jinay's regime graphs, optional baseline results

In [ ]:
import torch
N_GPU = torch.cuda.device_count(); print('torch', torch.__version__, '| GPUs:', N_GPU); assert N_GPU >= 1, 'Set Accelerator to GPU T4 x2'
if not REPO.exists():
    r = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/SYLan2019/DSTAGNN.git', str(REPO)], capture_output=True, text=True)
    assert REPO.exists(), 'git clone failed: turn Internet ON. ' + r.stderr

def find(name):
    pat = '/kaggle/input/**/%s' % name if ON_KAGGLE else './**/%s' % name
    h = [p for p in glob.glob(pat, recursive=True) if (ON_KAGGLE or 'DSTAGNN' not in p) and '/working/runs/' not in p]   # on Kaggle only /kaggle/input is searched, so a dataset folder named DSTAGNN is fine
    return h[0] if h else None

# data
f = find(DATASET + '.npz'); assert f, 'Add the %s dataset (PEMS data with %s.npz) via Add Data.' % (DATASET, DATASET)
raw = np.load(f)['data']; assert raw.shape == EXPECT[DATASET], 'bad shape %s' % (raw.shape,)
ddir = REPO / 'data' / DATASET; ddir.mkdir(parents=True, exist_ok=True); shutil.copy(f, ddir / (DATASET + '.npz'))
if not (ddir / (DATASET + '.csv')).exists():
    c = find(DATASET + '.csv'); assert c, 'Missing %s.csv (road edges), add a dataset that has it' % DATASET; shutil.copy(c, ddir / (DATASET + '.csv'))

# Jinay's graphs
g = find('strg_001_%s_weekday_peak.csv' % DATASET)
if g is None:
    z = find('jinay_graphs_outputs.zip')
    if z: zipfile.ZipFile(z).extractall(WORK / 'jinay'); g = find('strg_001_%s_weekday_peak.csv' % DATASET) or str(next((WORK / 'jinay').rglob('strg_001_%s_weekday_peak.csv' % DATASET), '')) or None
assert g, "Could not find Jinay's regime graphs. Add his output dataset (it contains regime_graphs/%s/strg_001_%s_weekday_peak.csv)." % (DATASET, DATASET)
GRAPH_DIR = Path(g).parent.parent; print('regime graphs from:', GRAPH_DIR)
meta = json.load(open(GRAPH_DIR / 'meta.json')); START_DOW = int(meta['start_dow'][DATASET]); print('start day-of-week (0=Mon):', START_DOW)
if DATASET == 'PEMS04' and START_DOW != 0: print('*** WARNING: PEMS04 must use start day-of-week 0 (confirmed from the data). This graph dataset is the OLD one with offset %d: use the newest version of tcstag-regime-graphs. ***' % START_DOW)
if DATASET != 'PEMS04':      # authors ship only PEMS04 graphs: use Jinay's global graph, exactly as in his PEMS08 baseline
    for kind in ['stag', 'strg']:
        shutil.copy(GRAPH_DIR / DATASET / ('%s_001_%s_global.csv' % (kind, DATASET)), ddir / ('%s_001_%s.csv' % (kind, DATASET)))
for nm in ['global', 'weekday_peak', 'weekday_offpeak', 'weekend']:
    assert (GRAPH_DIR / DATASET / ('strg_001_%s_%s%s.csv' % (DATASET, nm, GRAPH_K))).exists(), 'missing graph %s%s' % (nm, GRAPH_K)

# baseline results (optional): Harshith (PEMS04) / Jinay (PEMS08)
bname = 'baseline_summary.csv' if DATASET == 'PEMS04' else 'baseline_summary_%s.csv' % DATASET
bpath = find(bname); BASE = pd.read_csv(bpath) if bpath else None
BASE_PREDS = sorted(glob.glob(str(Path(bpath).parent / 'runs' / ('%s_AG_seed*' % DATASET) / 'test_preds.npz'))) if bpath else []
print('baseline results:', 'FOUND (%d seeds, %d pred files)' % (len(BASE), len(BASE_PREDS)) if BASE is not None else 'not attached: comparison vs baseline will be skipped')

r = subprocess.run([sys.executable, 'prepareData.py', '--config', 'configurations/%s_dstagnn.conf' % DATASET], cwd=REPO, capture_output=True, text=True)
assert (ddir / ('%s_r1_d0_w0_dstagnn.npz' % DATASET)).exists(), 'prepareData failed: ' + r.stderr[-800:]
print('data prepared')

In [ ]:
!find /kaggle/input -iname "*baseline*"

In [ ]:
!ls /kaggle/input/datasets/*/*

## Cell 3: Write the model and the runner
`model/tcstag.py` is our model (a thin extension of the authors' classes). `runner_tc.py` trains one (dataset, variant, seed) with the same protocol as the baseline runner: Huber loss, Adam 1e-4, batch 32, early stopping on validation loss, best-validation checkpoint tested.

In [ ]:
TCSTAG = r'''# TC-STAG: time-conditioned graph prior + time embeddings on top of the authors' DSTAGNN (model/DSTAGNN_my.py)
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from model.DSTAGNN_my import DSTAGNN_block, DSTAGNN_submodule, cheb_conv_withSAt
from lib.utils import scaled_Laplacian, cheb_polynomial

STEPS_PER_DAY = 288
# component order of the graph stack: 0 = global, 1 = weekday_peak, 2 = weekday_offpeak, 3 = weekend
def regime_index(t, start_dow):
    tod = t % STEPS_PER_DAY; dow = (t // STEPS_PER_DAY + start_dow) % 7
    peak = ((tod >= 84) & (tod < 120)) | ((tod >= 192) & (tod < 228))          # 7-10h, 16-19h
    return torch.where(dow >= 5, torch.full_like(t, 3), torch.where(peak, torch.full_like(t, 1), torch.full_like(t, 2)))

class TCChebConv(cheb_conv_withSAt):
    # authors' graph convolution, except T_k may be per-sample (B,N,N): the graph SUPPORT itself changes with time
    def forward(self, x, spatial_attention, adj_pa, T_mix=None):
        batch_size, N, _, num_t = x.shape; outputs = []
        for ts in range(num_t):
            gs = x[:, :, :, ts]; output = torch.zeros(batch_size, N, self.out_channels).to(self.DEVICE)
            for k in range(self.K):
                T_k = self.cheb_polynomials[k] if T_mix is None else T_mix[:, k]
                att = F.softmax(spatial_attention[:, k, :, :] + adj_pa.mul(self.mask[k]), dim=1)
                rhs = T_k.mul(att).permute(0, 2, 1).matmul(gs)
                output = output + rhs.matmul(self.Theta[k])
            outputs.append(output.unsqueeze(-1))
        return self.relu(torch.cat(outputs, dim=-1))

class TimeEncoder(nn.Module):               # time-of-day + day-of-week embedding
    def __init__(self, dim, start_dow):
        super().__init__(); self.start_dow = start_dow
        self.tod = nn.Embedding(STEPS_PER_DAY, dim); self.dow = nn.Embedding(7, dim)
    def forward(self, t):
        return self.tod(t % STEPS_PER_DAY) + self.dow((t // STEPS_PER_DAY + self.start_dow) % 7)

class GraphMixer(nn.Module):
    # produces the attention-prior graph A_STRG(t). modes: hard | soft_static | soft_time
    def __init__(self, graphs, mode, start_dow, cheb=None, gate_dim=16):
        super().__init__()
        self.mode, self.start_dow = mode, start_dow
        self.register_buffer('graphs', graphs)                           # (C, N, N), C = 4 : attention priors A_STRG^(c)
        self.has_cheb = cheb is not None
        if self.has_cheb: self.register_buffer('cheb', cheb)             # (C, K, N, N): Chebyshev polynomials of each component's STAG
        C = graphs.shape[0]
        if mode == 'soft_static':
            self.logits = nn.Parameter(torch.zeros(C))
        elif mode == 'soft_time':
            self.e_tod = nn.Embedding(STEPS_PER_DAY, gate_dim); self.e_dow = nn.Embedding(7, gate_dim)
            self.mlp = nn.Sequential(nn.Linear(2 * gate_dim, 32), nn.ReLU(), nn.Linear(32, C))
    def post_init(self):                                                # called AFTER the repo's init loop (which overwrites biases)
        if self.mode == 'soft_static':
            nn.init.constant_(self.logits, 0.0); self.logits.data[0] = 2.0   # start close to the global graph (softmax ~ 0.71 / 0.10 each)
        elif self.mode == 'soft_time':
            nn.init.zeros_(self.mlp[2].weight); nn.init.zeros_(self.mlp[2].bias); self.mlp[2].bias.data[0] = 2.0
    def weights(self, t):                                               # (B, C)
        C = self.graphs.shape[0]
        if self.mode == 'hard':
            return F.one_hot(regime_index(t, self.start_dow), C).float()
        if self.mode == 'soft_static':
            return F.softmax(self.logits, 0).unsqueeze(0).expand(t.shape[0], -1)
        h = torch.cat([self.e_tod(t % STEPS_PER_DAY), self.e_dow((t // STEPS_PER_DAY + self.start_dow) % 7)], -1)
        return F.softmax(self.mlp(h), -1)
    def forward(self, t):                                               # -> prior (B,N,N), Chebyshev stack (B,K,N,N) or None
        w = self.weights(t)
        prior = torch.einsum('bc,cnm->bnm', w, self.graphs)
        T_mix = torch.einsum('bc,ckmn->bkmn', w, self.cheb) if self.has_cheb else None
        return prior, T_mix

class TCBlock(DSTAGNN_block):
    # identical to the authors' block, except: per-sample graph prior and an optional time embedding added in the spatial attention input
    def forward(self, x, res_att, adj_pa_t=None, te=None, T_mix=None):
        batch_size, num_of_vertices, num_of_features, num_of_timesteps = x.shape
        TEmx = self.EmbedT(x, batch_size) if num_of_features == 1 else x.permute(0, 2, 3, 1)
        TATout, re_At = self.TAt(TEmx, TEmx, TEmx, None, res_att)
        x_TAt = self.pre_conv(TATout.permute(0, 2, 3, 1))[:, :, :, -1].permute(0, 2, 1)
        SEmx_TAt = self.dropout(self.EmbedS(x_TAt, batch_size))
        if te is not None:
            SEmx_TAt = SEmx_TAt + te.unsqueeze(1)
        STAt = self.SAt(SEmx_TAt, SEmx_TAt, None)
        spatial_gcn = self.cheb_conv_SAt(x, STAt, self.adj_pa if adj_pa_t is None else adj_pa_t, T_mix)
        X = spatial_gcn.permute(0, 2, 1, 3)
        time_conv = self.fcmy(torch.cat([self.gtu3(X), self.gtu5(X), self.gtu7(X)], dim=-1))
        time_conv_output = self.relu(time_conv) if num_of_features == 1 else self.relu(X + time_conv)
        x_residual = self.residual_conv(x.permute(0, 2, 1, 3)) if num_of_features == 1 else x.permute(0, 2, 1, 3)
        x_residual = self.ln(F.relu(x_residual + time_conv_output).permute(0, 3, 2, 1)).permute(0, 2, 3, 1)
        return x_residual, re_At

class TCSubmodule(DSTAGNN_submodule):
    def __init__(self, *args, variant_cfg=None, graphs=None, cheb=None, start_dow=0, **kw):
        super().__init__(*args, **kw)
        d_model = args[14]                               # positional args of DSTAGNN_submodule
        for b in self.BlockList: b.__class__ = TCBlock; b.cheb_conv_SAt.__class__ = TCChebConv
        self.use_te, self.gmode = variant_cfg['te'], variant_cfg['graph']
        if self.use_te:
            self.time_enc = TimeEncoder(d_model, start_dow)
            self.te_head = nn.Linear(d_model, 128)
            self.fuse = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 128))
        self.mixer = GraphMixer(graphs, self.gmode, start_dow, cheb if variant_cfg.get('cheb', True) else None) if self.gmode != 'none' else None
        self.to(self.DEVICE)
    def post_init(self):
        if self.use_te: nn.init.zeros_(self.fuse[2].weight); nn.init.zeros_(self.fuse[2].bias)   # start as the baseline
        if self.mixer is not None: self.mixer.post_init()
    def forward(self, x, t):
        te = self.time_enc(t) if self.use_te else None
        adj_t, T_mix = self.mixer(t) if self.mixer is not None else (None, None)
        need_concat, res_att = [], 0
        for block in self.BlockList:
            x, res_att = block(x, res_att, adj_t, te, T_mix); need_concat.append(x)
        final_x = torch.cat(need_concat, dim=-1)
        out1 = self.final_conv(final_x.permute(0, 3, 1, 2))[:, :, :, -1].permute(0, 2, 1)       # (B, N, 128)
        if self.use_te:
            h = torch.cat([out1, self.te_head(te).unsqueeze(1).expand(-1, out1.shape[1], -1)], -1)
            out1 = out1 + self.fuse(h)
        return self.final_fc(out1)

VARIANTS = {   # name -> config
    'base':        dict(te=False, graph='none'),
    'te':          dict(te=True,  graph='none'),         # time embeddings only
    'hard':        dict(te=False, graph='hard'),         # regime graph chosen by the clock, no learned gate, no time embeddings
    'soft_static': dict(te=False, graph='soft_static'),  # learned global mixture weights, not time-dependent
    'te_hard':     dict(te=True,  graph='hard'),
    'tcstag':      dict(te=True,  graph='soft_time'),    # FULL METHOD (prior AND Chebyshev support are time-conditioned)
    'tcstag_prior':dict(te=True,  graph='soft_time', cheb=False),  # ablation: only the attention prior is time-conditioned (mostly inert, see paper notes)
}

def make_tc_model(DEVICE, variant, graphs, stags, start_dow, num_of_d, nb_block, in_channels, K, nb_chev_filter, nb_time_filter, time_strides,
                  adj_mx, adj_pa, adj_TMD, num_for_predict, len_input, num_of_vertices, d_model, d_k, d_v, n_heads):
    L_tilde = scaled_Laplacian(adj_mx)
    cheb = [torch.from_numpy(i).type(torch.FloatTensor).to(DEVICE) for i in cheb_polynomial(L_tilde, K)]
    g = torch.tensor(np.stack(graphs), dtype=torch.float32).to(DEVICE) if graphs is not None else None
    ch = None
    if stags is not None:                                                # (C, K, N, N): one Chebyshev stack per component graph
        ch = torch.stack([torch.stack([torch.from_numpy(i).type(torch.FloatTensor) for i in cheb_polynomial(scaled_Laplacian(a), K)]) for a in stags]).to(DEVICE)
    model = TCSubmodule(DEVICE, num_of_d, nb_block, in_channels, K, nb_chev_filter, nb_time_filter, time_strides, cheb, adj_pa, adj_TMD,
                        num_for_predict, len_input, num_of_vertices, d_model, d_k, d_v, n_heads,
                        variant_cfg=VARIANTS[variant], graphs=g, cheb=ch, start_dow=start_dow)
    for p in model.parameters():                                         # same init loop as the authors' make_model
        if p.dim() > 1: nn.init.xavier_uniform_(p)
        else: nn.init.uniform_(p)
    model.post_init()
    return model
'''
RUNNER = r'''# TC-STAG runner: one training run = (dataset, variant, seed). Same protocol as runner.py (baseline), plus timestamps and regime graphs.
import os, sys, json, time, argparse, random, configparser
p = argparse.ArgumentParser()
p.add_argument('--repo', default='.'); p.add_argument('--dataset', default='PEMS04')
p.add_argument('--variant', default='tcstag'); p.add_argument('--seed', type=int, default=1); p.add_argument('--gpu', default='0')
p.add_argument('--max_epochs', type=int, default=60); p.add_argument('--patience', type=int, default=15)
p.add_argument('--out', required=True); p.add_argument('--deadline', type=float, default=0.0)
p.add_argument('--graph_dir', default=''); p.add_argument('--graph_k', default='', help="'' = 1% graphs, '_k5' = 5% graphs")
p.add_argument('--graph', default='AG', help='AG = learned STAG for the Chebyshev conv (the paper model); G = static road graph (DSTAGNN-G)'); p.add_argument('--start_dow', type=int, default=0); p.add_argument('--batch_size', type=int, default=0, help='0 = config value (32)'); p.add_argument('--debug_batches', type=int, default=0)
args = p.parse_args()
os.environ['CUDA_VISIBLE_DEVICES'] = str(args.gpu)
os.chdir(args.repo); sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
if os.environ.get('TCSTAG_CPU_TEST'): torch.Tensor.cuda = lambda self, *a, **k: self     # CPU smoke tests only (repo hard-codes .cuda())
from model.tcstag import make_tc_model, VARIANTS
from lib.dataloader import load_weighted_adjacency_matrix, load_PA
from lib.utils1 import get_adjacency_matrix2
from lib.metrics import masked_mape_np

random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)
torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True
cfg = configparser.ConfigParser(); cfg.read('configurations/%s_dstagnn.conf' % args.dataset); D, T = cfg['Data'], cfg['Training']
ds, N = args.dataset, int(D['num_of_vertices']); bs = args.batch_size or int(T['batch_size']); lr = float(T['learning_rate'])
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu'); os.makedirs(args.out, exist_ok=True)

# ---- data (with timestamps) ----
f = np.load(D['graph_signal_matrix_filename'].replace('.npz', '') + '_r1_d0_w0_dstagnn.npz')
def loader(split, shuffle):
    x = torch.from_numpy(f[split + '_x'][:, :, 0:1, :]).float().to(DEVICE); y = torch.from_numpy(f[split + '_target']).float().to(DEVICE)
    t = torch.from_numpy(f[split + '_timestamp'][:, 0].astype(np.int64)).to(DEVICE)
    return DataLoader(TensorDataset(x, y, t), batch_size=bs, shuffle=shuffle), y
train_loader, _ = loader('train', True); val_loader, _ = loader('val', False); test_loader, test_target = loader('test', False)

# ---- graphs: component 0 = the baseline's own A_STRG; 1..3 = regime graphs (Jinay) ----
adj_mx = get_adjacency_matrix2(D['adj_filename'], N, id_filename=None)
adj_stag = load_weighted_adjacency_matrix(D['stag_filename'], N); adj_strg = load_PA(D['strg_filename'])
adj_merge = adj_mx if args.graph == 'G' else adj_stag     # exactly as the authors' train script (AG -> STAG)
print('Chebyshev graph for the base model:', 'road network (G)' if args.graph == 'G' else 'learned STAG (AG)', flush=True)
graphs = stags = None
if VARIANTS[args.variant]['graph'] != 'none':
    rd = lambda nm, kind='strg': (pd.read_csv(os.path.join(args.graph_dir, ds, '%s_001_%s_%s%s.csv' % (kind, ds, nm, args.graph_k)), header=None).values > 0).astype(np.float32)
    comp0 = adj_strg if args.graph_k == '' else rd('global')
    graphs = [comp0] + [rd(n) for n in ['weekday_peak', 'weekday_offpeak', 'weekend']]
    stag0 = adj_stag if args.graph_k == '' else rd('global', 'stag')            # component 0 of the Chebyshev graphs = the baseline's own STAG
    stags = [stag0] + [rd(n, 'stag') for n in ['weekday_peak', 'weekday_offpeak', 'weekend']]
    print('chebyshev graph neighbours per node (incl. self):', [float((s_ > 0).sum(1).mean()) for s_ in stags], flush=True)
    print('graph components, neighbours per node:', [float(g.sum(1).mean()) for g in graphs], flush=True)

net = make_tc_model(DEVICE, args.variant, graphs, stags, args.start_dow, int(T['in_channels']), int(T['nb_block']), int(T['in_channels']), int(T['K']),
                    int(T['nb_chev_filter']), int(T['nb_time_filter']), 1, adj_merge, adj_strg, adj_stag, int(D['num_for_predict']),
                    int(D['len_input']), N, int(T['d_model']), int(T['d_k']), int(T['d_k']), int(T['n_heads']))
n_params = int(sum(q.numel() for q in net.parameters())); print('variant', args.variant, '| params', n_params, '| device', DEVICE, flush=True)
crit = nn.SmoothL1Loss().to(DEVICE); opt = optim.Adam(net.parameters(), lr=lr)

def val_loss():
    net.eval(); tot = []
    with torch.no_grad():
        for i, (x, y, t) in enumerate(val_loader):
            if args.debug_batches and i >= args.debug_batches: break
            tot.append(crit(net(x, t), y).item())
    return float(np.mean(tot))

best_path = os.path.join(args.out, 'best.pt'); best_val, best_ep, bad, hist, stopped = float('inf'), -1, 0, [], 'max_epochs'; t_start = time.time()
for ep in range(args.max_epochs):
    t0 = time.time(); net.train(); tl = []
    for i, (x, y, t) in enumerate(train_loader):
        if args.debug_batches and i >= args.debug_batches: break
        opt.zero_grad(); loss = crit(net(x, t), y); loss.backward(); opt.step(); tl.append(loss.detach())
    tr = float(torch.stack(tl).mean()); vl = val_loss()
    hist.append(dict(epoch=ep, train_loss=tr, val_loss=vl, epoch_sec=time.time() - t0))
    improved = vl < best_val
    if improved: best_val, best_ep, bad = vl, ep, 0; torch.save(net.state_dict(), best_path)
    else: bad += 1
    print('ep %d train %.4f val %.4f %s  %.1fs' % (ep, tr, vl, '*' if improved else '', time.time() - t0), flush=True)
    if bad >= args.patience: stopped = 'early_stop'; break
    if args.deadline and time.time() > args.deadline: stopped = 'deadline'; break

net.load_state_dict(torch.load(best_path, map_location=DEVICE)); net.eval(); preds = []
with torch.no_grad():
    for i, (x, y, t) in enumerate(test_loader):
        if args.debug_batches and i >= args.debug_batches: break
        preds.append(net(x, t).cpu().numpy())
pred = np.concatenate(preds, 0); tgt = test_target.cpu().numpy()[:len(pred)]
def m(y, yh): return dict(MAE=float(np.mean(np.abs(y - yh))), RMSE=float(np.sqrt(np.mean((y - yh) ** 2))), MAPE=float(masked_mape_np(y, yh, 0)))
per_h = [m(tgt[:, :, i], pred[:, :, i]) for i in range(pred.shape[2])]; overall = m(tgt.reshape(-1), pred.reshape(-1))
np.savez_compressed(os.path.join(args.out, 'test_preds.npz'), prediction=pred.astype(np.float32), target=tgt.astype(np.float32), timestamp=f['test_timestamp'][:len(pred)])
if net.mixer is not None and net.mixer.mode != 'hard':      # learned gate weights for every (day-of-week, time-of-day): for the paper figure
    tt = torch.tensor([((d - args.start_dow) % 7) * 288 + s for d in range(7) for s in range(288)], device=DEVICE)
    with torch.no_grad(): gw = net.mixer.weights(tt).cpu().numpy().reshape(7, 288, -1)
    np.save(os.path.join(args.out, 'gate_weights.npy'), gw)
res = dict(dataset=ds, variant=args.variant, cheb_graph=args.graph, graph_k=args.graph_k, seed=args.seed, best_epoch=best_ep, best_val=best_val, epochs_run=len(hist),
           stopped_by=stopped, max_epochs=args.max_epochs, patience=args.patience, mean_epoch_sec=float(np.mean([h['epoch_sec'] for h in hist])),
           total_sec=time.time() - t_start, n_params=n_params, test_overall=overall, test_per_horizon=per_h, history=hist)
json.dump(res, open(os.path.join(args.out, 'results.json'), 'w'), indent=1); os.remove(best_path)
print('TEST overall:', overall, flush=True)
'''
(REPO / 'model' / 'tcstag.py').write_text(TCSTAG); (WORK / 'runner_tc.py').write_text(RUNNER)
print('written:', REPO / 'model' / 'tcstag.py', '|', WORK / 'runner_tc.py')

## Cell 4: Launch helper + 1-minute smoke test of every variant (fails fast before the long runs)

In [ ]:
def launch(variant, seed, gpu, max_epochs, patience, out, extra=()):
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, str(WORK / 'runner_tc.py'), '--repo', str(REPO), '--dataset', DATASET, '--variant', variant, '--seed', str(seed),
           '--gpu', str(gpu), '--max_epochs', str(max_epochs), '--patience', str(patience), '--out', str(out), '--deadline', str(SESSION_DEADLINE),
           '--graph_dir', str(GRAPH_DIR), '--graph_k', GRAPH_K, '--graph', GRAPH, '--start_dow', str(START_DOW)] + list(extra)
    return subprocess.run(cmd, stdout=open(str(out) + '.log', 'w'), stderr=subprocess.STDOUT)

if RUN_SMOKE_TEST:
    for v in ['te', 'tcstag', 'tcstag_prior', 'hard']:
        t0 = time.time(); out = RUNS / 'smoke' / v
        p = launch(v, 0, 0, 1, 1, out, extra=['--debug_batches', '3', '--batch_size', '8'])
        ok = p.returncode == 0 and (out / 'results.json').exists()
        print('%-12s %s  (%.0fs)' % (v, 'OK' if ok else 'FAILED', time.time() - t0))
        if not ok: print(open(str(out) + '.log').read()[-2500:]); raise SystemExit('Smoke test failed: send the traceback above to the group.')
    shutil.rmtree(RUNS / 'smoke', ignore_errors=True); print('all variants run forward + backward + test on GPU')

## Cell 5: Timing: one full epoch of the full model. Post the TIMING line in the group chat

In [ ]:
if RUN_TIMING_TEST:
    out = RUNS / 'timing_test'; p = launch('tcstag', 0, 0, 1, 1, out); assert p.returncode == 0, open(str(out) + '.log').read()[-2000:]
    sec = json.load(open(out / 'results.json'))['mean_epoch_sec']; shutil.rmtree(out, ignore_errors=True)
    n_jobs = len(JOBS); rounds = -(-n_jobs // N_GPU)
    print('>>> TIMING (%s, tcstag): %.0f s/epoch | %d jobs on %d GPU(s) = %d rounds | up to %.1f h per run | up to %.1f h wall clock' % (
        DATASET, sec, n_jobs, N_GPU, rounds, sec * MAX_EPOCHS / 3600, rounds * sec * MAX_EPOCHS / 3600))

## Cell 6: Train (one job per GPU in parallel; finished runs are skipped, so you can safely rerun)

In [ ]:
jobs = queue.Queue()
for v, s in JOBS: jobs.put((v, s))
status = {}
def rundir(v, s): return RUNS / ('%s_%s%s_seed%d' % (DATASET, v, GRAPH_K, s))
def worker(gpu):
    while True:
        try: v, s = jobs.get_nowait()
        except queue.Empty: return
        out = rundir(v, s)
        if (out / 'results.json').exists(): status[(v, s)] = 'skipped'; continue
        t0 = time.time(); p = launch(v, s, gpu, MAX_EPOCHS, PATIENCE, out)
        status[(v, s)] = 'ok' if p.returncode == 0 else 'FAILED (see %s.log)' % out
        print('%-12s seed %d on GPU %d -> %s in %.1f min' % (v, s, gpu, status[(v, s)], (time.time() - t0) / 60), flush=True)
th = [threading.Thread(target=worker, args=(g,)) for g in range(min(N_GPU, jobs.qsize()))]
def monitor(every=600):      # prints the newest epoch line of every job every 10 min, so the Logs tab of a committed run shows progress
    while any(t.is_alive() for t in th):
        time.sleep(every)
        for v, s in JOBS:
            lg = Path(str(rundir(v, s)) + '.log')
            ep_lines = [l for l in lg.read_text().splitlines() if l.startswith('ep ')] if lg.exists() else []
            if ep_lines: print('[monitor %s] %-12s seed %d | %s' % (time.strftime('%H:%M'), v, s, ep_lines[-1].strip()), flush=True)
[t.start() for t in th]
time.sleep(120)
for v, s in JOBS: print(open(str(rundir(v, s)) + '.log').read()[:600])
threading.Thread(target=monitor, daemon=True).start(); [t.join() for t in th]; print(status)

## Cell 7: Results table: every variant vs the baseline (mean ± std over seeds)

In [ ]:
rows, hrows = [], []
for fpath in sorted(glob.glob(str(RUNS / ('%s_*seed*/results.json' % DATASET)))):
    r = json.load(open(fpath)); o = r['test_overall']
    rows.append(dict(variant=r['variant'], seed=r['seed'], MAE=o['MAE'], MAPE=o['MAPE'], RMSE=o['RMSE'], best_epoch=r['best_epoch'], epochs_run=r['epochs_run'], stopped_by=r['stopped_by'], sec_per_epoch=r['mean_epoch_sec'], params=r['n_params']))
    hrows += [dict(variant=r['variant'], seed=r['seed'], horizon=i + 1, **h) for i, h in enumerate(r['test_per_horizon'])]
res = pd.DataFrame(rows)
if BASE is not None: res = pd.concat([res, BASE.assign(variant='baseline')[res.columns.intersection(BASE.columns).tolist() + ['variant']]], ignore_index=True)
print(res.round(3).sort_values(['variant', 'seed']).to_string(index=False))

order_v = ['baseline', 'base', 'te', 'hard', 'soft_static', 'te_hard', 'tcstag']
agg = res.groupby('variant')[['MAE', 'MAPE', 'RMSE']].agg(['mean', 'std', 'count']).reindex([v for v in order_v if v in res.variant.unique()])
summary = pd.DataFrame({'n_seeds': agg[('MAE', 'count')].astype(int)})
for m in ['MAE', 'MAPE', 'RMSE']: summary[m] = agg[(m, 'mean')].round(3).astype(str) + ' ± ' + agg[(m, 'std')].fillna(0).round(3).astype(str)
if 'baseline' in agg.index:
    for m in ['MAE', 'RMSE']: summary['d%s_vs_base_%%' % m] = (100 * (agg[(m, 'mean')] / agg.loc['baseline', (m, 'mean')] - 1)).round(2)
print('\n', summary.to_string())
from scipy import stats
if 'baseline' in res.variant.unique():
    print('\nWelch t-test on MAE vs baseline (INDICATIVE ONLY: 3 seeds is tiny; look at effect size vs seed std):')
    b = res[res.variant == 'baseline'].MAE.values
    for v in [x for x in order_v[1:] if x in res.variant.unique()]:
        a = res[res.variant == v].MAE.values
        if len(a) > 1 and len(b) > 1: print('  %-12s mean diff %+.3f  p=%.3f  (%d/%d seeds below baseline mean)' % (v, a.mean() - b.mean(), stats.ttest_ind(a, b, equal_var=False).pvalue, (a < b.mean()).sum(), len(a)))
summary.to_csv(WORK / 'tcstag_results_summary.csv'); res.to_csv(WORK / 'tcstag_all_runs.csv', index=False); pd.DataFrame(hrows).to_csv(WORK / 'tcstag_per_horizon.csv', index=False)
if {'te', 'tcstag'} <= set(res.variant):
    d = res[res.variant == 'tcstag'].MAE.mean() - res[res.variant == 'te'].MAE.mean()
    print('\nKEY QUESTION: does the graph mixture add anything on top of time embeddings?  tcstag - te = %+.3f MAE' % d,
          '| seed std of te: %.3f' % res[res.variant == 'te'].MAE.std())

## Cell 8: Error by regime: does the graph help where we claim (rush hours / weekends)?
Uses the same regime definition as the baseline notebooks. Compare `tcstag` against `te`, not only against the baseline: the gain from the graph mixture is the difference between those two rows.

In [ ]:
def regime_of(t):
    tod = t % 288; dow = ((t // 288) + START_DOW) % 7
    peak = ((tod >= 84) & (tod < 120)) | ((tod >= 192) & (tod < 228))
    return np.where(dow >= 5, 'weekend', np.where(peak, 'weekday_peak', 'weekday_offpeak'))
recs = []
files = [(('baseline', int(Path(p).parent.name.split('seed')[1])), p) for p in BASE_PREDS] + \
        [((json.load(open(Path(p).parent / 'results.json'))['variant'], json.load(open(Path(p).parent / 'results.json'))['seed']), p) for p in sorted(glob.glob(str(RUNS / ('%s_*seed*/test_preds.npz' % DATASET))))]
for (v, s), p in files:
    z = np.load(p); reg = regime_of(z['timestamp'][:, 0].astype(int))
    for name in ['weekday_peak', 'weekday_offpeak', 'weekend']:
        m = reg == name; e = z['prediction'][m] - z['target'][m]
        recs.append(dict(variant=v, seed=s, regime=name, MAE=np.abs(e).mean()))
reg_df = pd.DataFrame(recs)
if len(reg_df):
    pv = reg_df.groupby(['variant', 'regime']).MAE.mean().unstack().reindex([v for v in order_v if v in reg_df.variant.unique()])[['weekday_peak', 'weekday_offpeak', 'weekend']]
    print(pv.round(3).to_string())
    for ref in ['baseline', 'base', 'te']:
        if ref in pv.index: print('\n%% change vs %s:' % ref); print((100 * (pv / pv.loc[ref] - 1)).round(2).drop(ref).to_string())
    reg_df.to_csv(WORK / 'tcstag_by_regime.csv', index=False)

## Cell 9: What the gate learned (a figure for the paper)

In [ ]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
gws = [np.load(p) for p in sorted(glob.glob(str(RUNS / ('%s_tcstag%s_seed*/gate_weights.npy' % (DATASET, GRAPH_K)))))]
if gws:
    G = np.mean(gws, axis=0)                                   # (7 dow, 288 tod, 4 components), averaged over seeds
    names = ['global', 'weekday peak', 'weekday off-peak', 'weekend']; hrs = np.arange(288) / 12
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
    for a, (title, days) in zip(ax, [('Weekdays (Mon-Fri)', list(range(5))), ('Weekend (Sat-Sun)', [5, 6])]):
        # gate arrays are indexed by true day-of-week (0 = Monday)
        sel = G[days].mean(0)
        for c in range(4): a.plot(hrs, sel[:, c], label=names[c])
        a.set_title(title); a.set_xlabel('hour of day'); a.set_xlim(0, 24)
    ax[0].set_ylabel('learned mixture weight'); ax[1].legend(fontsize=8); plt.tight_layout(); plt.savefig(WORK / 'gate_weights_over_day.png', dpi=160); plt.show()
    spread = G.max(axis=(0, 1)) - G.min(axis=(0, 1))
    print('range of each component weight over the week:', dict(zip(names, spread.round(3))))
    print('=> if all ranges are near 0 the gate stayed at its initial value (the model ignored time-conditioning): report that honestly.')
else:
    print('no tcstag runs yet')

## Cell 10: Package and send

In [ ]:
for f_ in glob.glob(str(RUNS / '*/best.pt')): os.remove(f_)
shutil.rmtree(RUNS / 'timing_test', ignore_errors=True)
zp = WORK / 'advik_tcstag_outputs.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f_ in glob.glob(str(RUNS / '*/*')) + glob.glob(str(RUNS / '*.log')) + glob.glob(str(WORK / 'tcstag_*.csv')) + glob.glob(str(WORK / '*.png')):
        z.write(f_, os.path.relpath(f_, WORK))
print('zip:', zp, '| %.1f MB' % (os.path.getsize(zp) / 1e6))
print('Send to the group: the zip (Output tab), the TIMING line, the table from Cell 7 (incl. KEY QUESTION line), the regime table from Cell 8, the figure from Cell 9.')